<a href="https://colab.research.google.com/github/j-hay-214/osu-gradtda-5622-sp25/blob/main/_site/course_materials/hw/5/Hay_Jarrod_HW5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# GRADTDA5622 - Big Data Computing Foundations 2
## Homework 5: PySpark Practice
- Semester: Spring 2025
- Instructor: Brad Coy
- Section: 3716
- Student Name: Jarrod Hay
- Student Email: hay.177@osu.edu
- Student ID: 500287277
***

***
# Section: Overview
***

**The Objectives of This Assignment are:**
1. To practice using common Spark operations.
2. To practice using Spark to solve problems and answer questions.

**Overview:**
- I have provided a step by step approach you can follow.  Fill in the ... in each cell.
- I have filled in some cells for you, as examples.
- Refer to the **PySpark_DeepDive1** notebook covered in the **Deep Dive: Spark** module for examples of code that can be used in this assignment.

**Some Good Resources:**
- https://spark.apache.org/docs/latest/api/python/index.html
- https://spark.apache.org/docs/latest/api/python/reference/pyspark.pandas/frame.html
- https://sparkbyexamples.com/pyspark-tutorial/

**Instructions:**
- **Follow the instructions** in each section.
- **Fill in** the **Conclusions** section.

***
# Section: Setup
- Add any needed imports, helper functions, etc., here.
***

In [1]:
try:
    import pyspark
except:
    print('Installing pyspark')
    !pip install pyspark
    import pyspark

# try:
#     import pyspark_config
# except:
#     print('Installing pyspark_config')
#     !pip install pyspark_config
#     import pyspark_config

In [2]:
# NOTE: If any of these libraries are not already loaded on OSC Jupyter+Spark (e.g., seaborn),
#  go the the Launcher (New Launcher in the JupyterLab Files menu), open a Terminal, and type
#  "pip install seaborn" (or the needed library).
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sb
from time import time
from pyspark import SparkContext
from pyspark.sql import SparkSession
import pyspark.sql.functions as SqlF

pd.set_option('display.max_columns', 50) #include to avoid ... in middle of display
pyspark.__version__

'3.5.5'

In [3]:
spark = SparkSession.builder.master("local[*]") \
                    .appName('MyApp') \
                    .getOrCreate()
sc = spark.sparkContext  # Get the context, so we have a short name for it if we need it.
#print(sc.appName)

In [ ]:
# Identify the location of the shared data folder
shared_data_directory = "../shared_Sp23/"

***
# Section: 1 - Problem Overview / Business Understanding
***

The three provided datasets contain records of users' recommendations for movies (items).  The datasets are **data.csv**, **item.csv**, and **user.csv**.  See: https://grouplens.org/datasets/movielens/100k/ for descriptions of the datasets.

The **goal** of the exercise is to **estimate a rating** for the movie **"Mission: Impossible (1996)"** by **User15**.

- We will use a trivial approach (barely sensible, but easy):
  - Find all "other users" who have rated the Mission Impossible movie already.
  - If the average age of those "other reviewers" is within +- 20 years of User15's age:
    - Average the ratings those "other users" gave to Mission Impossible to estimate a rating for User15 for Mission Impossible.
    - Otherwise, just use the average rating User15 gave to other movies to estimate a rating for User15 for Mission Impossible.  

***
# Section: 2 - Data Understanding
***

***
## Section: 2.1 - Describe the meaning and type of data for each attribute.
- This can be pulled from the original metadata documentation, if available, fom other sources, or postulated based on values within the data.  Be explicit regarding the source, assumptions, etc., in particular if you are making educated guesses.
***

In [ ]:
# Insert code and/or commentary here...  EXAMPLE CODE BELOW...

At the highest level, this dataset consists of 100,000 ratings from 943 different users across 1,682 different movies. Each user rated at least 20 movies and rated each movie on a 1-5 scale. The data in question is divided into three files: data, item, and user.

The 'data' file is the full dataset of all 100,000 ratings. It consists of 4 different attributes: 'user_id', 'item_id', 'rating', and 'timestamp'. The schema for this dataset is printed in the first chunk below, which shows the data types for each attribute.

**'user_id'** is an integer attribute that provides a unique numerical ID for each separate user that submitted ratings for the dataset. The .txt file describing the data does not explicitly state that each user would have a unique identifier, but it is reasonable to assume that this would be the case.

**'item_id'** is an integer attribute that provides a unique numerical ID for each movie that received at least 1 rating from a user. Similar to the previous attribute, it is not explicitly stated that each movie has a unique identifier, but it is reasonable to assume that each movie would be identified uniquely as this would make the rating system unhelpful otherwise.

**'rating'** is an integer attribute ranging from 1 - 5 that represents the rating given by the user for that particular movie. Lower values represent a lower evaluation of the movie while higher values represent more positive evaluations of the movie, meaning that movies with a rating of 1 are considered to be the "worst" rated movies and those with a 5 are considered as the "best" ratings.

**'timestamp'** is an integer attribute that represents the time when that particular rating was submitted. This value represents the number of seconds that have occurred since 1/1/1970 UTC, which serves as a way of standardizing date/time data to be more easily analyzed.

### Read the three datasets.

In [4]:
# Reading in the 'data' dataset
data_df = spark.read.csv('data.csv', header=True, inferSchema=True).orderBy('user_id','item_id')
print(data_df.count())
data_df.printSchema()
data_df.show(2,truncate=False)

100000
root
 |-- user_id: integer (nullable = true)
 |-- item_id: integer (nullable = true)
 |-- rating: integer (nullable = true)
 |-- timestamp: integer (nullable = true)

+-------+-------+------+---------+
|user_id|item_id|rating|timestamp|
+-------+-------+------+---------+
|1      |1      |5     |874965758|
|1      |2      |3     |876893171|
+-------+-------+------+---------+
only showing top 2 rows



The second dataset in question is the 'item' dataset. This dataset provides more details about the movies being rated in the 'data' dataset. Essentially, this includes 6 attributes that provide context for the movie in question. These include: 'movie_id', 'movie_title', 'release_date', 'video_release_date', 'IMDb_URL', and movie genre, which has been divided into 19 binary attributes. The details are below.

**'movie_id'** is an integer attribute that provides a unique numerical ID for each movie that received at least 1 rating from a user. These values are equivalent to the 'item_id' values in the 'data' dataset. This provides an identifier that can allow one to connect the details under that 'movie_id' to their associated ratings under the corresponding 'item_id'.

**'movie_title'** is a string attribute that provides the actual title of the movie written out as a line of text.

**'release_date'** is a string attribute that provides the date that the movie was released into theatres. Rather than using the standardized date/time transformation that was used above, this data is written as DD-Month-YYYY with Month being the appreviated version of the month of video release.

**'video_release_date'** is a string attribute providing the date that the movie was released to video for people to begin watching at home. However, all values of this attribute are Null, meaning that the data was either not recorded or a movie did not release to video. The code snippet below the item_df schema shows an attempt to filter the dataframe for only values that are not null, and it shows that this resulting dataframe is empty.

**'IMDb_URL'** is a string attribute that provides the URL for the IMDb website for that particular movie. This URL could be pasted into a web browser to visit the IMDb site for that movie.

Movie genre is a collection of 19 different binary attributes, which include 'unknown', 'Action', 'Adventure', 'Animation', 'Childrens', 'Comedy', 'Crime', 'Documentary', 'Drama', 'Fantasy', 'Film_Noir', 'Horror', 'Musical', 'Mystery', 'Romance', 'Sci_Fi', 'Thriller', 'War', and 'Western'. All 19 of these attributes are integer type data with values of 0 or 1 with 0 indicating that the genre in question does not describe that particular movie. A value of 1 indicates that this is a fitting genre for the movie. Having the data set up in this way allows for movies to fit into multiple genres as multiple genre attributes can be marked as 1.

In [15]:
item_df = spark.read.csv('item.csv', header=True, inferSchema=True).orderBy('movie_id')
print(item_df.count())
item_df.printSchema()
item_df.show(2, truncate=False)

1682
root
 |-- movie_id: integer (nullable = true)
 |-- movie_title: string (nullable = true)
 |-- release_date: string (nullable = true)
 |-- video_release_date: string (nullable = true)
 |-- IMDb_URL: string (nullable = true)
 |-- unknown: integer (nullable = true)
 |-- Action: integer (nullable = true)
 |-- Adventure: integer (nullable = true)
 |-- Animation: integer (nullable = true)
 |-- Childrens: integer (nullable = true)
 |-- Comedy: integer (nullable = true)
 |-- Crime: integer (nullable = true)
 |-- Documentary: integer (nullable = true)
 |-- Drama: integer (nullable = true)
 |-- Fantasy: integer (nullable = true)
 |-- Film_Noir: integer (nullable = true)
 |-- Horror: integer (nullable = true)
 |-- Musical: integer (nullable = true)
 |-- Mystery: integer (nullable = true)
 |-- Romance: integer (nullable = true)
 |-- Sci_Fi: integer (nullable = true)
 |-- Thriller: integer (nullable = true)
 |-- War: integer (nullable = true)
 |-- Western: integer (nullable = true)

+--------+

In [14]:
# Selecting only rows where video_release_date is not a Null value
item_df2 = item_df.where(item_df.video_release_date.isNotNull()).show(2)

+--------+-----------+------------+------------------+--------+-------+------+---------+---------+---------+------+-----+-----------+-----+-------+---------+------+-------+-------+-------+------+--------+---+-------+
|movie_id|movie_title|release_date|video_release_date|IMDb_URL|unknown|Action|Adventure|Animation|Childrens|Comedy|Crime|Documentary|Drama|Fantasy|Film_Noir|Horror|Musical|Mystery|Romance|Sci_Fi|Thriller|War|Western|
+--------+-----------+------------+------------------+--------+-------+------+---------+---------+---------+------+-----+-----------+-----+-------+---------+------+-------+-------+-------+------+--------+---+-------+
+--------+-----------+------------+------------------+--------+-------+------+---------+---------+---------+------+-----+-----------+-----+-------+---------+------+-------+-------+-------+------+--------+---+-------+



Lastly, there is the 'user_df' dataset. This includes demographic information collected for each of the users that provided ratings in the 'data' dataset. This includes 5 attributes, which include: 'user_id', 'age', 'gender', 'occupation', and 'zip_code'.

**'user_id'** is an integer attribute that provides a unique numerical ID for each separate user that submitted ratings for the dataset. This is the same as the 'user_id' attribute from the 'data' dataset, allowing for the details of the user to be connected to their ratings in the master list.

**'age'** is an integer attribute that provides the age of the user at the time that the data was pulled.

**'gender'** is a string attribute that provides the user's gender. This could either be entered as 'M' for male or 'F' for female.

**'occupation'** is a string attribute providing the user's occupation.

**'zip_code'** is a string attribute providing the zip code of the user's address.

In [16]:
user_df = spark.read.csv('user.csv', header=True, inferSchema=True).orderBy('user_id')
print(user_df.count())
user_df.printSchema()
user_df.show(2, truncate=False)

943
root
 |-- user_id: integer (nullable = true)
 |-- age: integer (nullable = true)
 |-- gender: string (nullable = true)
 |-- occupation: string (nullable = true)
 |-- zip_code: string (nullable = true)

+-------+---+------+----------+--------+
|user_id|age|gender|occupation|zip_code|
+-------+---+------+----------+--------+
|1      |24 |M     |technician|85711   |
|2      |53 |F     |other     |94043   |
+-------+---+------+----------+--------+
only showing top 2 rows



***
## Section: 2.2 - Provide basic statistics for the attributes.
- For example: counts, percentiles, mean, median, standard deviation. The statistics should be relevant for the type of attribute.
***

In [ ]:
data_df.describe().toPandas()

,summary,user_id,item_id,rating,timestamp
0,count,100000,100000,100000,100000
1,mean,462.48475,425.53013,3.52986,8.8352885148862E8
2,stddev,266.61442012750877,330.7983563255847,1.1256735991443154,5343856.189502826
3,min,1,1,1,874724710
4,max,943,1682,5,893286638


In [17]:
item_df_desc = (item_df
    .agg(SqlF.sum('unknown').alias('Unknown'),
         SqlF.count('movie_id').alias('Movies Total')


,summary,movie_id,movie_title,release_date,video_release_date,IMDb_URL,unknown,Action,Adventure,Animation,Childrens,Comedy,Crime,Documentary,Drama,Fantasy,Film_Noir,Horror,Musical,Mystery,Romance,Sci_Fi,Thriller,War,Western
0,count,1682,1682,1681,0,1679,1682,1682,1682,1682,1682,1682,1682,1682,1682,1682,1682,1682,1682,1682,1682,1682,1682,1682,1682
1,mean,841.5,None,None,None,None,0.0011890606420927466,0.1492271105826397,0.0802615933412604,0.02497027348394768,0.07253269916765755,0.30023781212841855,0.06480380499405469,0.029726516052318668,0.43103448275862066,0.013079667063020214,0.014268727705112961,0.054696789536266346,0.03329369797859691,0.03626634958382877,0.14684898929845422,0.06004756242568371,0.1492271105826397,0.04221165279429251,0.01605231866825208
2,stddev,485.69589250888254,None,None,None,None,0.034472500474212284,0.35641816109515484,0.2717785571305444,0.15608088423430846,0.25944503383362605,0.4584976014014335,0.2462525621467922,0.16988233706244027,0.4953681954046321,0.11364976236574524,0.11863177582846385,0.2274550708557175,0.17945577207322622,0.1870077359968219,0.35406057857880563,0.2376455954301871,0.3564181610951548,0.20113149982303108,0.1257141110346824
3,min,1,'Til There Was You (1997),1/1/1922,None,http://us.imdb.com/M/title-exact/Independence%...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
4,max,1682,� k�ldum klaka (Cold Fever) (1994),9/6/1996,None,http://us.imdb.com/Title?Yao+a+yao+yao+dao+wai...,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1


In [ ]:
"unknown_tot"),
         SqlF.sum('Action').alias("action_tot"),
         SqlF.sum('Adventure').alias("adv_tot"),
         SqlF.sum('Animation').alias("anim_tot"),
         SqlF.sum('Childrens').alias("child_tot"),
         SqlF.sum('Comedy').alias("com_tot"),
         SqlF.sum('Crime').alias("action_tot"),
         SqlF.sum('Action').alias("action_tot"),
         SqlF.sum('Action').alias("action_tot"),
         SqlF.sum('Action').alias("action_tot"),
         SqlF.sum('Action').alias("action_tot"),
         SqlF.sum('Action').alias("action_tot"),
         SqlF.sum('Action').alias("action_tot"),
         SqlF.sum('Action').alias("action_tot"),
         SqlF.sum('Action').alias("action_tot"),
         SqlF.sum('Action').alias("action_tot"),
         SqlF.sum('Action').alias("action_tot"),
         SqlF.sum('Action').alias("action_tot"),
         SqlF.sum('Action').alias("action_tot"),

In [ ]:
user_df...

***
# Section: 3 - Data Pre-Processing
***

In [ ]:
# Trim the data_df and item_df datasets down to only the necessary information.
# Consider using "select" to keep only the 'user_id','item_id','rating' columns for data_df,
#   and the 'movie_id' and 'movie_title' columns for item_df.

data_df = ...
item_df = ...
#user_df = no changes needed

In [ ]:
# Calculate the user statistics (count, min, average, max ratings).
# Consider using "groupBy", "agg", "orderBy", etc.
# Create a Dataframe containing: |user_id|count_rating|min_rating|avg_rating|max_rating|

user_rating_df = ...

print(user_rating_df.count())
user_rating_df.show(5,truncate=False)

In [ ]:
# Calculate the movie statistics (count, min, average, max ratings).
# Consider using "groupBy", "agg", "orderBy", etc.
# Create a Dataframe containing: |item_id|count_rating|min_rating|avg_rating|max_rating|

item_rating_df = ...

print(item_rating_df.count())
item_rating_df.show(5,truncate=False)

***
# Section: 4 - Recommendation System
- For each of the steps below, I have provided an outline of the computation to perform and the expected output structure.
- Please fill in the computations.
- You may choose to deviate from this structure, but if you do so, decribe the steps you chose.
***

In [ ]:
# Specify the user and movie of interest.

user_x_id = 15
movie_y_title = "Mission: Impossible (1996)"

In [ ]:
# Get the demographics of this user, and save the age and gender.
# Consider using the user_df from above, and the "filter" and "collect" operations.

user_x_demographics = ...
user_x_demographics.show()

user_x_age = user_x_demographics.collect()[0]['age']
print("user_x_age:",user_x_age)

user_x_gender = ...
print("user_x_gender:",user_x_gender)

In [ ]:
# Get user X's average rating for all movies they actually have rated.
# Consider using the user_rating_df from above, and the "filter" and "collect" operations.

user_x_avg_rating = ...

print("user_x_avg_rating:",user_x_avg_rating)

In [ ]:
# Get the movie id for this movie title.
# Consider using the "filter" and "collect" operations.

movie_y_id = ...

print("movie_y_id:",movie_y_id)

In [ ]:
# Get all of the other users who have rated movie Y.
# Consider using "filter", "select", "orderBy", "withColumnRenamed".
# Create a Dataframe containing: |other_user_id|movie_y_rating|

other_reviewers_df = ...

print("other_reviewers_df.count:",other_reviewers_df.count())
other_reviewers_df.show(5,truncate=False)

In [ ]:
# For each of the other reviewers of movie_y, get the demographics.
# Consider using the other_reviewers_df and user_df from above, and the "join" operation.
# Create a Dataframe containing: |other_user_id|movie_y_rating|age|gender|occupation|zip_code|

other_reviewer_demo_df = ...

print("other_reviewer_demo_df.count:",other_reviewer_demo_df.count())
other_reviewer_demo_df.show(5,truncate=False)

In [ ]:
# For the other reviewers, get the average movie_y_rating and average age.
# Consider using the other_reviewer_demo_df and the "agg", "SqlF.avg" and "collect" operations.
# Create a Dataframe containing: |item_id|other_user_id|other_user_rating|user_x_rating|

avg_other_reviewer_movie_y_rating = ...
print("avg_other_reviewer_movie_y_rating:",avg_other_reviewer_movie_y_rating)

avg_other_reviewer_age = ...
print("avg_other_reviewer_age:",avg_other_reviewer_age)

In [ ]:
# This is a trivial way to make a recommendation.  Normally we would do something much
# more sophisticated.  But we will keep it simple here.

# If the average age of the other reviewers is within +- 20 years of user_x age,
#  then assume user_x's rating of movie_y will be the average rating given by the other reviewers.
#  Otherwise, assume user_x's rating for movie_y will be the average rating user_x has given
#  to other movies they have rated.
# Print this rating, with the explanation.

age_diff = ...
print("age_diff:",age_diff)

if age_diff <= 20:
    print('''The average age of reviewers of movie_y is within 20 years
    of the age of user_x.  So we will use their average rating for movie_y:''',avg_other_reviewer_movie_y_rating)
else:
    print('''The average age of reviewers of movie_y is NOT within 20 years
    of the age of user_x.  So we will use the average rating of user_x for other movies :''',user_x_avg_rating)

***
# Section: 6 - Conclusions
- What are your overall conclusions about the assignment?
- What did you learn?
***

In [ ]:
# Insert commentary here.